# Module 05, 06 & 07 Assignment
## Understanding the ML Problem, Exploratory Data Analysis (EDA) and Basic Preprocessing
**Dataset:** Titanic – Machine Learning from Disaster (Kaggle)

Download the **train.csv** file from Kaggle's Titanic competition and upload it to this notebook. Rename it to `titanic.csv` or update the file name in the loading cell.  
Dataset link: https://www.kaggle.com/datasets/yasserh/titanic-dataset

Total Marks: **100**

### Instructions
- This assignment covers:
  - **Module 05:** Basic ML problem framing (features, target, task type).
  - **Module 06:** Exploratory Data Analysis (EDA).
  - **Module 07:** Basic Preprocessing (handling missing values, encoding, scaling).
- Answer all questions inside this notebook using code and markdown.
- Do not delete the original question texts.
- At the end, the notebook should run from top to bottom without errors.

In [ ]:
# ==============================
# Setup
# ==============================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

print("Libraries imported.")

---
## Part 0: Understanding the ML Problem (Module 05) – 10 marks

### 0.1 Load the Dataset and Inspect (5 marks)
**Tasks:**
1. Load `titanic.csv` into a pandas DataFrame.
2. Show `.head()`, `.info()`, and `.describe()`.
3. Briefly look at the column names and data types.

In [ ]:
# 0.1 Load the Dataset and Inspect
# TODO: Change file name if needed.

df = pd.read_csv("titanic.csv")

print("Shape of dataset:", df.shape)
display(df.head())

print("\nInfo:")
df.info()

print("\nDescribe (numeric columns):")
display(df.describe())

### 0.2 Identify Features, Target and Task Type (5 marks)
**Tasks:**  
Based on the dataset:
1. Identify the **target variable** for a predictive model.
2. List at least **5 potential feature columns** you could use.
3. State whether this is a **classification** or **regression** problem, and explain **why**.

Write your answers below.

#### **Your answers**
- **Target variable:** `Survived` — a binary column (0 = did not survive, 1 = survived) that we want to predict.
- **Feature columns (at least 5):** `Pclass`, `Sex`, `Age`, `SibSp`, `Fare`, `Embarked`, `Parch`
- **Is this classification or regression, and why?:**  
  This is a **binary classification** problem because the target variable `Survived` takes only two discrete values (0 or 1). The goal is to assign each passenger to one of two classes: survived or not survived. Since the output is a category (not a continuous quantity), regression would be inappropriate; we need a classifier that models the probability of belonging to each class.

---
## Part A: Exploratory Data Analysis (EDA) – 45 marks

### 1. Initial Exploration and Cleaning Decisions (10 marks)
**Tasks:**
1. Show the number of unique values in each column.
2. Identify columns that are clearly **IDs or high-cardinality text** (for example, `PassengerId`, `Name`, `Ticket`).
3. Decide which of these columns you will **drop** for the rest of the analysis and justify in 2–3 sentences.

In [ ]:
# 1. Initial Exploration and Cleaning Decisions (Task 1 is given)
print("Unique values per column:")
for col in df.columns:
    print(f"  {col:15s}: {df[col].nunique()} unique values")

#### Columns to drop and justification
- **Columns dropped:** `PassengerId`, `Name`, `Ticket`
- **Justification:** `PassengerId` is a simple row index with no predictive value — it is just an arbitrary number assigned to each passenger. `Name` is a high-cardinality text field; while titles embedded in names can be engineered into features, the raw name itself uniquely identifies individuals and adds noise rather than signal. Similarly, `Ticket` has near-unique values (most tickets are different), making it nearly an identifier with no direct relationship to survival probability.

In [ ]:
# Drop high-cardinality / ID columns
df.drop(columns=["PassengerId", "Name", "Ticket"], inplace=True)
print("Columns after dropping:", df.columns.tolist())
print("New shape:", df.shape)

---
### 2. Univariate Analysis (15 marks)
**Tasks:**
- Plot histograms for numeric features: `Age`, `Fare`, `SibSp`, `Parch`.
- Plot countplots for categorical features: `Sex`, `Pclass`, `Embarked`.
- Write **two to three insights** about the distributions and any obvious patterns.

In [ ]:
# 2. Univariate Analysis

numeric_cols = ["Age", "Fare", "SibSp", "Parch"]

# Histograms code is written for you
df[numeric_cols].hist(bins=20, figsize=(10, 6))
plt.suptitle("Histograms of Numeric Features", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

# Countplots for key categorical variables
cat_cols = ["Sex", "Pclass", "Embarked"]
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, col in zip(axes, cat_cols):
    order = sorted(df[col].dropna().unique())
    sns.countplot(data=df, x=col, order=order, ax=ax, hue=col, palette="Set2", legend=False)
    ax.set_title(f"Countplot of {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

#### Write Your insights
- **Insight 1:** The `Age` distribution is roughly bell-shaped and centered around 28–30 years, but it has a slight right skew, with a small peak near age 0–5 suggesting a number of infants/young children on board. About 20% of age values are missing and need to be imputed.
- **Insight 2:** `Fare` is heavily right-skewed — the vast majority of passengers paid a relatively low fare (under £30), while a small number of first-class passengers paid very high fares (up to £500+). This suggests a log transformation could be useful before modelling.
- **Insight 3:** Males outnumber females roughly 2:1, and the majority of passengers were in 3rd class (`Pclass = 3`). Most passengers embarked from Southampton (`S`), with Cherbourg (`C`) and Queenstown (`Q`) being less common ports.

---
### 3. Bivariate Analysis with Target (15 marks)
Use `Survived` as the target variable.

**Tasks:**
- Compute and plot a **correlation heatmap** for numeric features, including `Survived`.
- Create a **pairplot** for: `Age`, `Fare`, `SibSp`, `Parch`, and `Survived`.
- Write **two to three insights**, including which variables seem associated with survival.

In [ ]:
# 3. Bivariate Analysis with Target

num_for_corr = ["Survived", "Age", "Fare", "SibSp", "Parch"]

# Correlation Heatmap
plt.figure(figsize=(8, 6))
corr_matrix = df[num_for_corr].corr()
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm",
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title("Correlation Heatmap of Numeric Features vs Survived")
plt.tight_layout()
plt.show()

# Pairplot
pair_df = df[num_for_corr].copy()
pair_df["Survived"] = pair_df["Survived"].astype(str).map({"0": "Not Survived", "1": "Survived"})
g = sns.pairplot(pair_df, hue="Survived", plot_kws={"alpha": 0.5},
                 palette={"Not Survived": "tomato", "Survived": "steelblue"})
g.fig.suptitle("Pairplot of Numeric Features Colored by Survival", y=1.02)
plt.show()

#### Your insights
- **Insight 1:** `Fare` has the strongest positive correlation with `Survived` (~+0.26). Passengers who paid higher fares (mostly 1st-class) had significantly better survival chances — likely because they had better access to lifeboats and were housed in upper deck cabins.
- **Insight 2:** `Age` shows a weak negative correlation with survival (~−0.07). Younger passengers, especially children, had slightly higher survival rates (the "women and children first" policy), but the overall effect is modest.
- **Insight 3:** In the pairplot, the `Fare` vs `Survived` scatter clearly separates the two classes: survived passengers tend to cluster at higher fare values, while non-survivors dominate the low-fare region — further confirming that socioeconomic status was a key survival factor.

---
### 4. Categorical vs Target Analysis (5 marks)
**Tasks:**  
For each of the following categorical features: `Sex`, `Pclass`, `Embarked`:
- Plot a **bar chart** showing the proportion of passengers who survived in each category.
- Write **two short insights** about which categories have higher or lower survival chances.

In [ ]:
# 4. Categorical vs Target Analysis [Bar chart code is written for you]

cat_target_cols = ["Sex", "Pclass", "Embarked"]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, c in zip(axes, cat_target_cols):
    ct = pd.crosstab(df[c], df["Survived"], normalize="index")
    ct.columns = ["Not Survived", "Survived"]
    ct.plot(kind="bar", stacked=True, ax=ax, color=["tomato", "steelblue"])
    ax.set_title(f"Survival Proportion by {c}")
    ax.set_ylabel("Proportion")
    ax.set_xlabel(c)
    ax.tick_params(axis="x", rotation=0)
    ax.legend(loc="upper right", fontsize=8)
    print(f"\nSurvival proportion by {c}:")
    display(ct)

plt.tight_layout()
plt.show()

#### Your insights
- **Insight 1:** Female passengers had a dramatically higher survival rate (approximately 74%) compared to males (approximately 19%), reflecting the "women and children first" evacuation policy enforced by the crew during the sinking.
- **Insight 2:** Survival rate decreases sharply with passenger class — 1st class passengers survived at roughly 63%, 2nd class at ~47%, and 3rd class at only ~24%. This pattern reflects the physical layout of the ship (3rd class was in lower decks, farther from lifeboats) and the socioeconomic bias in rescue prioritisation.

---
## Part B: Basic Data Preprocessing – 45 marks

Focus: **Handling missing values, encoding categorical variables, and scaling numeric features.**

### 5. Handling Missing Values (10 marks)
**Tasks:**
1. Show the count of missing values in each column.
2. Decide how to handle missing values for:
   - `Age` (numeric)
   - `Embarked` (categorical)
   - `Cabin` (many missing values)
3. Implement your chosen strategy in code.
4. Show missing value counts again to confirm.
5. Explain your choices in **3–4 sentences**.

In [ ]:
# 5. Handling Missing Values

print("Missing values before:")  # [Task 1 is done for you]
print(df.isna().sum())

# Drop Cabin — too many missing values to impute meaningfully
df = df.drop(columns=["Cabin"])

# Fill Age with the median (robust to outliers)
df["Age"] = df["Age"].fillna(df["Age"].median())

# Fill Embarked with the mode (most frequent port)
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

print("\nMissing values after handling:")
print(df.isna().sum())
print("\nTotal remaining missing values:", df.isna().sum().sum())

#### Explanation of your strategy
- **Why you dropped `Cabin`:** The `Cabin` column has approximately 77% missing values, which is far too large a proportion to impute reliably — any imputation strategy would largely fabricate data and could introduce misleading patterns. Dropping it entirely is the safest approach.
- **Why you used median for `Age`:** `Age` is a numeric feature with a right-skewed distribution and some high outliers (elderly passengers). The **median** is preferred over the mean in such cases because it is robust to outliers and better represents the "typical" passenger age, avoiding artificial inflation of imputed values.
- **Why you used mode for `Embarked`:** `Embarked` is a categorical variable with only 2 missing values out of 891. Filling these with the **mode** (the most frequent embarkation port, which is 'S' — Southampton) is the simplest and most appropriate strategy — it introduces minimal bias given how few values are missing.

---
### 6. Encoding Categorical Variables (15 marks)
Focus on the following categorical features:
- `Sex`
- `Embarked`
- `Pclass` (treat as categorical)

**Tasks:**
- Use `OneHotEncoder` from sklearn to encode these features.
- Keep `Survived` and the main numeric features (`Age`, `Fare`, `SibSp`, `Parch`).
- Show the shape and head of the encoded DataFrame.

In [ ]:
# 6. Encoding Categorical Variables

df_enc = df.copy()

# Select features for encoding and keep important numeric ones
cat_features = ["Sex", "Embarked", "Pclass"]
numeric_features = ["Age", "Fare", "SibSp", "Parch"]
target_col = "Survived"

# One-hot encode categorical variables
ohe = OneHotEncoder(sparse_output=False, drop="first", dtype=float)
encoded_array = ohe.fit_transform(df_enc[cat_features].astype(str))
encoded_df = pd.DataFrame(
    encoded_array,
    columns=ohe.get_feature_names_out(cat_features),
    index=df_enc.index
)

# Build final encoded DataFrame
df_model = pd.concat(
    [df_enc[[target_col] + numeric_features], encoded_df],
    axis=1
)

print("Shape after encoding:", df_model.shape)
display(df_model.head())
print("\nColumn names:", df_model.columns.tolist())

---
### 7. Scaling Numeric Features (15 marks)
**Tasks:**
- Use `StandardScaler` on the numeric features: `Age`, `Fare`, `SibSp`, `Parch`.
- Plot boxplots of these features **before** and **after** scaling.
- Write **two sentences** explaining how scaling changes the numeric values and why it is useful before training some ML models.

In [ ]:
# 7. Scaling Numeric Features

score_cols = ["Age", "Fare", "SibSp", "Parch"]

# Boxplot before scaling
plt.figure(figsize=(8, 4))
sns.boxplot(data=df_model[score_cols], palette="Set3")
plt.title("Numeric Features BEFORE Scaling")
plt.ylabel("Original Values")
plt.tight_layout()
plt.show()

# Apply StandardScaler
scaler = StandardScaler()
df_scaled = df_model.copy()
df_scaled[score_cols] = scaler.fit_transform(df_model[score_cols])

# Boxplot after scaling
plt.figure(figsize=(8, 4))
sns.boxplot(data=df_scaled[score_cols], palette="Set3")
plt.title("Numeric Features AFTER StandardScaler")
plt.ylabel("Standardised Values (z-score)")
plt.tight_layout()
plt.show()

print("Mean of scaled columns (should be ~0):")
print(df_scaled[score_cols].mean().round(4))
print("\nStd of scaled columns (should be ~1):")
print(df_scaled[score_cols].std().round(4))

#### Your observations on scaling
- **Sentence 1:** After applying `StandardScaler`, each numeric feature is transformed so that it has a mean of approximately 0 and a standard deviation of 1, making features measured on very different scales (e.g., `Fare` in pounds vs `Parch` in counts) directly comparable on the same axis in the boxplot.
- **Sentence 2:** Scaling is essential for distance-based and gradient-descent algorithms (such as K-Nearest Neighbours, Logistic Regression, Support Vector Machines, and Neural Networks) because without it, features with larger numeric ranges (like `Fare`) would dominate the model, causing it to effectively ignore features with smaller ranges (like `SibSp` or `Parch`).

---
### 8. Simple Feature Quality Check (5 marks)
**Tasks:**
- Show `.info()` of the final processed DataFrame (`df_scaled`).
- Confirm there are no missing values.
- Confirm that all remaining features are numeric and suitable for most ML models.
- Write **one short concluding remark** about dataset readiness.

In [ ]:
# 8. Simple Feature Quality Check
print("=== Final Dataset Info ===")
print(df_scaled.info())

print("\nTotal missing values in final DataFrame:", df_scaled.isna().sum().sum())

print("\nData types in final DataFrame:")
print(df_scaled.dtypes)

print("\nAre all feature columns numeric?", 
      all(df_scaled.drop(columns=["Survived"]).dtypes != object))

print("\nFinal shape:", df_scaled.shape)
print("Sample (first 3 rows):")
display(df_scaled.head(3))

#### Final remark on dataset readiness
- **Your remark:** The final processed DataFrame `df_scaled` has **no missing values**, all features are **numeric** (float64/int64), and the numeric columns have been **standardised** to zero mean and unit variance. The dataset is now fully prepared — with encoding, imputation, and scaling complete — and is ready to be fed directly into most supervised machine learning algorithms such as Logistic Regression, Decision Trees, or Support Vector Machines.

---
## Submission
- Run all cells from top to bottom to make sure everything works.
- Set the Colab file's shareable link to 'Anyone with the link' and 'View' access, then submit it in the Phitron Assignment module's Assignment submission section.